# V23 bounded pair-field ranker

Development-only locked fit. This notebook does not need the 100 GB competition dataset. Attach the `v23_cfar_pair_field_v1_kaggle.zip` upload as a Kaggle dataset, enable a T4 GPU and Internet, then run the cells in order.

The fit remains read-only: no assignment, graph mutation, full-199 evaluation, or submission is authorized.

In [ ]:
from pathlib import Path
import glob, json, os, shutil, subprocess, sys, time

BRANCH = 'v23-detector-native-evidence'
EXPECTED_COMMIT = 'fb3156a6fdb779990ac385a6d23f621926ffe857'
ROOT = Path('/kaggle/working/Atabey')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/drosadocastro-bit/Atabey.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', EXPECTED_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == EXPECTED_COMMIT, (actual_commit, EXPECTED_COMMIT)
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = f"{ROOT / 'src'}:{ROOT / 'scripts'}"
sys.path.insert(0, str(ROOT / 'src'))
print('Atabey commit verified:', actual_commit)

In [ ]:
EXPECTED = {
    'actions_sha256': '39c49efdf3c93d2ac6c791e85db4c26fe79bf13bf34c76b27107ffa54096a72f',
    'events_sha256': 'd09275c827126adf869815d8c907c09f1e143a2f1d3f57cf70721110b4bcb1c2',
    'parents_sha256': 'd81aeb498135e87caf6ebffa05d7d6d3d3e8b50359762ff418e81c0b3c8c981d',
}
matches = []
for path in Path('/kaggle/input').rglob('manifest.json'):
    try:
        manifest = json.loads(path.read_text())
    except Exception:
        continue
    if manifest.get('status') == 'v23_locked_pair_field_dataset' and all(manifest.get(k) == v for k, v in EXPECTED.items()):
        matches.append(path.parent)
assert len(matches) == 1, f'Expected exactly one locked V23 dataset, found: {matches}'
DATASET_ROOT = matches[0]
manifest = json.loads((DATASET_ROOT / 'manifest.json').read_text())
assert manifest['parent_fields'] == 54
assert manifest['events'] == 29
assert manifest['actions'] == 2264
assert len(list((DATASET_ROOT / 'parents').glob('*.npy'))) == 54
print('Locked dataset:', DATASET_ROOT)
print({key: manifest[key] for key in ('parent_fields', 'events', 'actions')})

In [ ]:
# Kaggle may transparently unpack nested .gz members from an uploaded ZIP.
# Reconstruct the exact locked byte layout in writable storage.
import gzip, hashlib

SOURCE_ROOT = DATASET_ROOT
STAGED_ROOT = Path('/kaggle/working/v23_cfar_pair_field_v1')
if STAGED_ROOT.exists():
    shutil.rmtree(STAGED_ROOT)
(STAGED_ROOT / 'parents').mkdir(parents=True)
shutil.copy2(SOURCE_ROOT / 'manifest.json', STAGED_ROOT / 'manifest.json')
shutil.copy2(SOURCE_ROOT / 'parents.json', STAGED_ROOT / 'parents.json')
parent_rows = json.loads((SOURCE_ROOT / 'parents.json').read_text())
for row in parent_rows:
    relative = Path(row['relative_path'])
    source = SOURCE_ROOT / relative
    if not source.exists():
        candidates = list(SOURCE_ROOT.parent.rglob(relative.name))
        assert len(candidates) == 1, (relative, candidates)
        source = candidates[0]
    shutil.copy2(source, STAGED_ROOT / relative)

for stem, expected_hash in (
    ('actions', EXPECTED['actions_sha256']),
    ('events', EXPECTED['events_sha256']),
):
    target = STAGED_ROOT / f'{stem}.jsonl.gz'
    gzip_source = SOURCE_ROOT / f'{stem}.jsonl.gz'
    plain_source = SOURCE_ROOT / f'{stem}.jsonl'
    if gzip_source.exists():
        shutil.copy2(gzip_source, target)
    else:
        if not plain_source.exists():
            candidates = list(SOURCE_ROOT.parent.rglob(f'{stem}.jsonl'))
            assert len(candidates) == 1, (stem, candidates)
            plain_source = candidates[0]
        with plain_source.open('rb') as source, target.open('wb') as raw:
            with gzip.GzipFile(filename='', mode='wb', fileobj=raw, mtime=0) as zipped:
                shutil.copyfileobj(source, zipped)
    actual_hash = hashlib.sha256(target.read_bytes()).hexdigest()
    assert actual_hash == expected_hash, (stem, actual_hash, expected_hash)

DATASET_ROOT = STAGED_ROOT
print('Kaggle-normalized locked dataset:', DATASET_ROOT)
print('All 54 tensors and both deterministic gzip hashes verified')

In [ ]:
import numpy as np
import scipy
import torch

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before fitting'
subprocess.run([sys.executable, '-m', 'pytest', str(ROOT / 'tests/test_pair_field_ranker.py'), str(ROOT / 'tests/test_v23_bounded_pair_field_ranker_preregistration.py'), '-q'], check=True, env=RUN_ENV)

In [ ]:
from atabey.tracking.pair_field_ranker import (
    assemble_action_field, build_pair_field_ranker,
    load_locked_pair_field_data, model_parameter_count,
)

locked = load_locked_pair_field_data(DATASET_ROOT)
model = build_pair_field_ranker().cuda().eval()
field = assemble_action_field(locked, locked.actions[0])
with torch.inference_mode():
    probe_score = model(torch.from_numpy(field[None]).cuda()).item()
assert model_parameter_count(model) == 20145
assert field.shape == (5, 33, 33, 33)
print('Hash-verified loader and CUDA forward pass ready')
print('Probe score (unfitted, not evidence):', probe_score)
del model
torch.cuda.empty_cache()

## Locked fit

This is the long-running cell. It fits three frozen seeds across three outer folds and all preregistered controls. Completed seed/fold shards are written atomically. If the notebook disconnects but the Kaggle runtime survives, reconnect and rerun this cell; it automatically adds `--resume`.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/v23_pair_field_ranker_v1')
SUMMARY = Path('/kaggle/working/v23_bounded_pair_field_ranker_summary.json')
REPORT = Path('/kaggle/working/V23_BOUNDED_PAIR_FIELD_RANKER_RESULTS.md')
command = [
    sys.executable, '-u', str(ROOT / 'scripts/fit_v23_bounded_pair_field_ranker.py'),
    '--dataset', str(DATASET_ROOT),
    '--contract', str(ROOT / 'tests/fixtures/v23_bounded_pair_field_ranker.json'),
    '--output-dir', str(OUTPUT_DIR),
    '--summary', str(SUMMARY),
    '--report', str(REPORT),
    '--device', 'cuda',
    '--batch-size', '16',
    '--maximum-epochs', '60',
]
if OUTPUT_DIR.exists():
    command.append('--resume')
    print('Resume mode enabled; completed fold shards will be reused')
started = time.time()
subprocess.run(command, check=True, env=RUN_ENV)
print('Locked fit completed in hours:', (time.time() - started) / 3600.0)

In [ ]:
result = json.loads(SUMMARY.read_text())
print('Decision:', result['decision'])
print('Passing seeds:', result['passing_seeds'])
print('Catastrophic seed present:', result['catastrophic_seed_present'])
for seed_result in result['seed_results']:
    pooled = seed_result['metrics']['main']['pooled']
    print(seed_result['seed'], {key: pooled[key] for key in ('recall_at_10', 'mrr', 'pairwise_accuracy')}, 'pass=', seed_result['gates']['passed_all'])
print()
print('Report:', REPORT)

In [ ]:
BUNDLE = Path('/kaggle/working/v23_pair_field_ranker_outputs')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
shutil.copytree(OUTPUT_DIR, BUNDLE / OUTPUT_DIR.name)
shutil.copy2(SUMMARY, BUNDLE / SUMMARY.name)
shutil.copy2(REPORT, BUNDLE / REPORT.name)
shutil.make_archive(str(BUNDLE), 'zip', BUNDLE)
print('Download:', str(BUNDLE) + '.zip')